# Score your method on flyvis-circuit-bench

Each of the 604 connection classes (all synapses from one cell type to another) of 10 trained flyvis models was
switched off, and the change of the network response is the ground truth. A method gives every class a score
(larger = more important) **without** looking at that truth. This notebook downloads the measurement package
(38.5 MB, sha256-verified), scores an example method and puts it next to the baselines of the
[leaderboard](https://github.com/natiixnt/flyvis-circuit-bench#leaderboard). It needs no GPU and runs in a few
minutes.

Replace `my_method` below with your own idea.

In [ ]:
# In Colab the package is not installed yet: install it from GitHub (numpy, scipy, matplotlib only).
import importlib.util
import subprocess
import sys

if importlib.util.find_spec("flyvis_circuit_bench") is None:
    url = "git+https://github.com/natiixnt/flyvis-circuit-bench"
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", url], check=True)

In [ ]:
# Download and verify the measurement package from the GitHub Release (skipped when it is already there).
# FCB_ARCHIVE and FCB_SUMS point to a local copy of the release files instead (offline use, tests).
import os
from pathlib import Path

from flyvis_circuit_bench import download
from flyvis_circuit_bench.data import load

DATA = Path("data/measurements-v1.0.0")
if not DATA.is_dir():
    if os.environ.get("FCB_ARCHIVE"):
        sums = Path(os.environ["FCB_SUMS"]).read_text(encoding="utf-8")
        download.install(Path(os.environ["FCB_ARCHIVE"]), sums, DATA.parent)
    else:
        download.download("natiixnt", DATA.parent)
data = load(DATA)
print("models:", sorted(data["models"]))

## What a method may use

`method_inputs` gives, for every model, the source and target cell type of each class, its synapse count, the
sum of |weights| of the trained model and the number of neuron-to-neuron edges. Nothing derived from the
ablations is included. Methods that need activity or gradients can load the flyvis models themselves
(`pip install flyvis`, see the repository docs).

In [ ]:
from flyvis_circuit_bench import score

inputs = score.method_inputs(data)
x = inputs["000"]
for i in range(5):
    print(f"{x['source'][i]:>6} -> {x['target'][i]:<6} synapses {x['synapse_count'][i]:8.0f}  "
          f"sum |w| {x['sum_abs_weight'][i]:9.4f}  edges {x['n_edges'][i]:5.0f}")

## Your method

One score per connection class and model. The example ranks classes by the mean |weight| per edge.

In [ ]:
def my_method(x):
    return x["sum_abs_weight"] / x["n_edges"]


scores = {model: my_method(x) for model, x in inputs.items()}
result = score.evaluate(scores, data)
print(score.leaderboard_row("mean |weight| per edge", result, cost="trained weights"))

## Next to the baselines

The same metrics for the two anatomy baselines of the leaderboard. AUROC is the primary metric (95% CI over the
10 models); share of signal = (AUROC - 0.5) / (ceiling - 0.5); precision@61 = how many of the 61 most important
classes are in the method's top 61.

In [ ]:
from flyvis_circuit_bench.report import LEADERBOARD_HEAD

rows = {
    "synapse count": ({m: x["synapse_count"] for m, x in inputs.items()}, "connectome only"),
    "sum of |weights|": ({m: x["sum_abs_weight"] for m, x in inputs.items()}, "trained weights"),
    "mean |weight| per edge": (scores, "trained weights"),
}
results = {name: (score.evaluate(s, data), cost) for name, (s, cost) in rows.items()}
print("\n".join(LEADERBOARD_HEAD))
for name, (res, cost) in sorted(results.items(), key=lambda kv: -kv[1][0]["AUROC"]["pooled"]["mean"]):
    print(score.leaderboard_row(name, res, cost=cost))

## Submit

Save the scores with `numpy.savez("scores.npz", **scores)`, check them with
`fcb score --data data/measurements-v1.0.0 --scores scores.npz --name "your method"`, and open a pull request
that adds the printed row and a link to your code. See
[CONTRIBUTING.md](https://github.com/natiixnt/flyvis-circuit-bench/blob/main/CONTRIBUTING.md#add-a-method-to-the-leaderboard).